# Customer Engagement & Product Utilization Analytics for Retention Strategy

## Notebook 03: Feature Engineering & Customer Segmentation

### Notebook Objective

The objective of this notebook is to create business-friendly customer segments and analytical features that help evaluate customer engagement, product utilization, financial value, and retention strength.

The cleaned dataset from Notebook 02 will be used as the base dataset.

---

### Why Feature Engineering is Important

The original dataset contains useful customer information, but many business insights require derived features.

For example:

- `IsActiveMember` can be converted into an easy-to-read engagement status.
- `Exited` can be converted into churn status.
- `Balance` can be grouped into balance segments.
- `EstimatedSalary` can be grouped into salary segments.
- `NumOfProducts` can be converted into product depth segments.
- Engagement, product usage, tenure, and financial value can be combined into a relationship strength score.

---

### Key Features to Create

In this notebook, we will create:

- Engagement Status
- Churn Status
- Credit Card Status
- Product Depth Segment
- Balance Segment
- Salary Segment
- Age Segment
- Tenure Segment
- High Value Customer Flag
- High-Value Disengaged Flag
- At-Risk Premium Customer Flag
- Relationship Strength Index
- Relationship Strength Tier

---

### Final Output

At the end of this notebook, we will export the feature-engineered dataset to:

`data/processed/european_bank_features.csv`

### Import Libraries

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)
print("Imported successfully")

Imported successfully


### Define Paths

In [3]:
# Define project paths
BASE_DIR = Path.cwd().parent

CLEANED_DATA_PATH = BASE_DIR / "data" / "cleaned" / "european_bank_cleaned.csv"
PROCESSED_DATA_DIR = BASE_DIR / "data" / "processed"
FEATURES_DATA_PATH = PROCESSED_DATA_DIR / "european_bank_features.csv"

print("Cleaned Data Path:", CLEANED_DATA_PATH)
print("Processed Data Path:", FEATURES_DATA_PATH)

Cleaned Data Path: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\cleaned\european_bank_cleaned.csv
Processed Data Path: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\processed\european_bank_features.csv


### Load Cleaned Dataset

In [4]:
# Load cleaned dataset
bank_df = pd.read_csv(CLEANED_DATA_PATH)

# Preview dataset
bank_df.head()

,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,"101,348.88",1
1,2025,15647311,Hill,608,Spain,Female,41,1,"83,807.86",1,0,1,"112,542.58",0
2,2025,15619304,Onio,502,France,Female,42,8,"159,660.80",3,1,0,"113,931.57",1
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,"93,826.63",0
4,2025,15737888,Mitchell,850,Spain,Female,43,2,"125,510.82",1,1,1,"79,084.10",0


### Create Feature Engineering Copy

In [5]:
# Create a feature engineering copy
features_df = bank_df.copy()

print(f"Feature engineering dataset created with {features_df.shape[0]:,} rows and {features_df.shape[1]:,} columns.")

Feature engineering dataset created with 10,000 rows and 14 columns.


### Create Basic Status Columns

In [6]:
# Create readable churn, engagement, and credit card status columns
features_df["Churn_Status"] = features_df["Exited"].map({
    0: "Retained",
    1: "Churned"
})

features_df["Engagement_Status"] = features_df["IsActiveMember"].map({
    0: "Inactive Member",
    1: "Active Member"
})

features_df["Credit_Card_Status"] = features_df["HasCrCard"].map({
    0: "No Credit Card",
    1: "Has Credit Card"
})

# Preview new status columns
features_df[
    [
        "CustomerId",
        "Exited",
        "Churn_Status",
        "IsActiveMember",
        "Engagement_Status",
        "HasCrCard",
        "Credit_Card_Status"
    ]
].head()

,CustomerId,Exited,Churn_Status,IsActiveMember,Engagement_Status,HasCrCard,Credit_Card_Status
0,15634602,1,Churned,1,Active Member,1,Has Credit Card
1,15647311,0,Retained,1,Active Member,0,No Credit Card
2,15619304,1,Churned,0,Inactive Member,1,Has Credit Card
3,15701354,0,Retained,0,Inactive Member,0,No Credit Card
4,15737888,0,Retained,1,Active Member,1,Has Credit Card


### Basic Status Feature Validation

In [7]:
# Validate newly created status columns
status_validation = {
    "Churn_Status": features_df["Churn_Status"].value_counts(),
    "Engagement_Status": features_df["Engagement_Status"].value_counts(),
    "Credit_Card_Status": features_df["Credit_Card_Status"].value_counts()
}

for feature, distribution in status_validation.items():
    print(f"\n{feature} Distribution:")
    print(distribution)


Churn_Status Distribution:
Churn_Status
Retained    7963
Churned     2037
Name: count, dtype: int64

Engagement_Status Distribution:
Engagement_Status
Active Member      5151
Inactive Member    4849
Name: count, dtype: int64

Credit_Card_Status Distribution:
Credit_Card_Status
Has Credit Card    7055
No Credit Card     2945
Name: count, dtype: int64


### Product Depth Segmentation

In [8]:
# Create product depth segment based on number of products
def assign_product_depth_segment(product_count):
    if product_count == 1:
        return "Single Product"
    elif product_count == 2:
        return "Optimal Product Depth"
    elif product_count >= 3:
        return "High Product Load"
    else:
        return "Unknown"

features_df["Product_Depth_Segment"] = features_df["NumOfProducts"].apply(assign_product_depth_segment)

# Product depth distribution
product_depth_distribution = (
    features_df["Product_Depth_Segment"]
    .value_counts()
    .reset_index()
)

product_depth_distribution.columns = ["Product Depth Segment", "Customer Count"]

product_depth_distribution["Customer %"] = (
    product_depth_distribution["Customer Count"] / len(features_df) * 100
).round(2)

product_depth_distribution

,Product Depth Segment,Customer Count,Customer %
0,Single Product,5084,50.84
1,Optimal Product Depth,4590,45.90
2,High Product Load,326,3.26


### Product Depth Segment vs Churn Validation

In [9]:
# Validate churn rate by product depth segment
product_depth_churn = (
    features_df
    .groupby("Product_Depth_Segment")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Churn_Rate=("Exited", "mean"),
        Avg_Num_Products=("NumOfProducts", "mean")
    )
    .reset_index()
)

product_depth_churn["Churn_Rate"] = (
    product_depth_churn["Churn_Rate"] * 100
).round(2)

product_depth_churn["Avg_Num_Products"] = (
    product_depth_churn["Avg_Num_Products"]
).round(2)

product_depth_churn = product_depth_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

product_depth_churn

,Product_Depth_Segment,Customer_Count,Churned_Customers,Churn_Rate,Avg_Num_Products
0,High Product Load,326,280,85.89,3.18
2,Single Product,5084,1409,27.71,1.00
1,Optimal Product Depth,4590,348,7.58,2.00


### Balance Segmentation

In [10]:
# Create balance segment based on account balance
def assign_balance_segment(balance):
    if balance == 0:
        return "Zero Balance"
    elif balance <= 50000:
        return "Low Balance"
    elif balance <= 100000:
        return "Medium Balance"
    elif balance <= 150000:
        return "High Balance"
    else:
        return "Premium Balance"

features_df["Balance_Segment"] = features_df["Balance"].apply(assign_balance_segment)

balance_segment_distribution = (
    features_df["Balance_Segment"]
    .value_counts()
    .reset_index()
)

balance_segment_distribution.columns = ["Balance Segment", "Customer Count"]

balance_segment_distribution["Customer %"] = (
    balance_segment_distribution["Customer Count"] / len(features_df) * 100
).round(2)

balance_segment_distribution

,Balance Segment,Customer Count,Customer %
0,High Balance,3830,38.30
1,Zero Balance,3617,36.17
2,Medium Balance,1509,15.09
3,Premium Balance,969,9.69
4,Low Balance,75,0.75


### Balance Segment vs Churn Validation

In [11]:
# Validate churn rate by balance segment
balance_segment_churn = (
    features_df
    .groupby("Balance_Segment")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Churn_Rate=("Exited", "mean"),
        Avg_Balance=("Balance", "mean")
    )
    .reset_index()
)

balance_segment_churn["Churn_Rate"] = (
    balance_segment_churn["Churn_Rate"] * 100
).round(2)

balance_segment_churn["Avg_Balance"] = (
    balance_segment_churn["Avg_Balance"]
).round(2)

balance_segment_churn = balance_segment_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

balance_segment_churn

,Balance_Segment,Customer_Count,Churned_Customers,Churn_Rate,Avg_Balance
1,Low Balance,75,26,34.67,"39,447.53"
0,High Balance,3830,987,25.77,"123,693.73"
3,Premium Balance,969,224,23.12,"166,767.67"
2,Medium Balance,1509,300,19.88,"83,867.11"
4,Zero Balance,3617,500,13.82,0.00


### Salary Segmentation

In [12]:
# Create salary segment based on estimated salary
def assign_salary_segment(salary):
    if salary <= 50000:
        return "Low Salary"
    elif salary <= 100000:
        return "Medium Salary"
    elif salary <= 150000:
        return "High Salary"
    else:
        return "Premium Salary"

features_df["Salary_Segment"] = features_df["EstimatedSalary"].apply(assign_salary_segment)

salary_segment_distribution = (
    features_df["Salary_Segment"]
    .value_counts()
    .reset_index()
)

salary_segment_distribution.columns = ["Salary Segment", "Customer Count"]

salary_segment_distribution["Customer %"] = (
    salary_segment_distribution["Customer Count"] / len(features_df) * 100
).round(2)

salary_segment_distribution

,Salary Segment,Customer Count,Customer %
0,High Salary,2555,25.55
1,Medium Salary,2537,25.37
2,Premium Salary,2455,24.55
3,Low Salary,2453,24.53


### Salary Segment vs Churn Validation

In [13]:
# Validate churn rate by salary segment
salary_segment_churn = (
    features_df
    .groupby("Salary_Segment")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Churn_Rate=("Exited", "mean"),
        Avg_Salary=("EstimatedSalary", "mean")
    )
    .reset_index()
)

salary_segment_churn["Churn_Rate"] = (
    salary_segment_churn["Churn_Rate"] * 100
).round(2)

salary_segment_churn["Avg_Salary"] = (
    salary_segment_churn["Avg_Salary"]
).round(2)

salary_segment_churn = salary_segment_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

salary_segment_churn

,Salary_Segment,Customer_Count,Churned_Customers,Churn_Rate,Avg_Salary
3,Premium Salary,2455,527,21.47,"175,443.73"
0,High Salary,2555,517,20.23,"124,900.25"
1,Low Salary,2453,489,19.93,"24,926.07"
2,Medium Salary,2537,504,19.87,"74,861.75"


### Age Segmentation

In [14]:
# Create age segment
def assign_age_segment(age):
    if age < 30:
        return "Young Customer"
    elif age < 45:
        return "Mid-Age Customer"
    elif age < 60:
        return "Mature Customer"
    else:
        return "Senior Customer"

features_df["Age_Segment"] = features_df["Age"].apply(assign_age_segment)

age_segment_distribution = (
    features_df["Age_Segment"]
    .value_counts()
    .reset_index()
)

age_segment_distribution.columns = ["Age Segment", "Customer Count"]

age_segment_distribution["Customer %"] = (
    age_segment_distribution["Customer Count"] / len(features_df) * 100
).round(2)

age_segment_distribution

,Age Segment,Customer Count,Customer %
0,Mid-Age Customer,6019,60.19
1,Mature Customer,1814,18.14
2,Young Customer,1641,16.41
3,Senior Customer,526,5.26


### Age Segment vs Churn Validation

In [15]:
# Validate churn rate by age segment
age_segment_churn = (
    features_df
    .groupby("Age_Segment")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Churn_Rate=("Exited", "mean"),
        Avg_Age=("Age", "mean")
    )
    .reset_index()
)

age_segment_churn["Churn_Rate"] = (
    age_segment_churn["Churn_Rate"] * 100
).round(2)

age_segment_churn["Avg_Age"] = (
    age_segment_churn["Avg_Age"]
).round(2)

age_segment_churn = age_segment_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

age_segment_churn

,Age_Segment,Customer_Count,Churned_Customers,Churn_Rate,Avg_Age
0,Mature Customer,1814,897,49.45,50.15
2,Senior Customer,526,147,27.95,66.63
1,Mid-Age Customer,6019,869,14.44,36.66
3,Young Customer,1641,124,7.56,25.93


### Tenure Segmentation

In [16]:
# Create tenure segment
def assign_tenure_segment(tenure):
    if tenure <= 2:
        return "New Customer"
    elif tenure <= 5:
        return "Developing Relationship"
    elif tenure <= 8:
        return "Established Relationship"
    else:
        return "Long-Term Relationship"

features_df["Tenure_Segment"] = features_df["Tenure"].apply(assign_tenure_segment)

tenure_segment_distribution = (
    features_df["Tenure_Segment"]
    .value_counts()
    .reset_index()
)

tenure_segment_distribution.columns = ["Tenure Segment", "Customer Count"]

tenure_segment_distribution["Customer %"] = (
    tenure_segment_distribution["Customer Count"] / len(features_df) * 100
).round(2)

tenure_segment_distribution

,Tenure Segment,Customer Count,Customer %
0,Established Relationship,3020,30.20
1,Developing Relationship,3010,30.10
2,New Customer,2496,24.96
3,Long-Term Relationship,1474,14.74


### Tenure Segment vs Churn Validation

In [17]:
# Validate churn rate by tenure segment
tenure_segment_churn = (
    features_df
    .groupby("Tenure_Segment")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Churn_Rate=("Exited", "mean"),
        Avg_Tenure=("Tenure", "mean")
    )
    .reset_index()
)

tenure_segment_churn["Churn_Rate"] = (
    tenure_segment_churn["Churn_Rate"] * 100
).round(2)

tenure_segment_churn["Avg_Tenure"] = (
    tenure_segment_churn["Avg_Tenure"]
).round(2)

tenure_segment_churn = tenure_segment_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

tenure_segment_churn

,Tenure_Segment,Customer_Count,Churned_Customers,Churn_Rate,Avg_Tenure
2,Long-Term Relationship,1474,314,21.30,9.33
3,New Customer,2496,528,21.15,1.25
0,Developing Relationship,3010,625,20.76,4.00
1,Established Relationship,3020,570,18.87,7.02


### High Value Customer Flag

In [18]:
# Define high-value customers using balance and salary thresholds
balance_threshold = features_df["Balance"].quantile(0.75)
salary_threshold = features_df["EstimatedSalary"].quantile(0.75)

features_df["High_Value_Customer_Flag"] = np.where(
    (features_df["Balance"] >= balance_threshold) |
    (features_df["EstimatedSalary"] >= salary_threshold),
    1,
    0
)

features_df["High_Value_Customer_Status"] = features_df["High_Value_Customer_Flag"].map({
    0: "Standard Value Customer",
    1: "High Value Customer"
})

print(f"Balance Threshold Used: {balance_threshold:,.2f}")
print(f"Salary Threshold Used: {salary_threshold:,.2f}")

features_df["High_Value_Customer_Status"].value_counts()

Balance Threshold Used: 127,644.24
Salary Threshold Used: 149,388.25


High_Value_Customer_Status
Standard Value Customer    5649
High Value Customer        4351
Name: count, dtype: int64

### High Value Customer vs Churn Validation

In [19]:
# Validate churn rate by high-value customer status
high_value_churn = (
    features_df
    .groupby("High_Value_Customer_Status")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Churn_Rate=("Exited", "mean"),
        Avg_Balance=("Balance", "mean"),
        Avg_Salary=("EstimatedSalary", "mean")
    )
    .reset_index()
)

high_value_churn["Churn_Rate"] = (
    high_value_churn["Churn_Rate"] * 100
).round(2)

high_value_churn["Avg_Balance"] = high_value_churn["Avg_Balance"].round(2)
high_value_churn["Avg_Salary"] = high_value_churn["Avg_Salary"].round(2)

high_value_churn = high_value_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

high_value_churn

,High_Value_Customer_Status,Customer_Count,Churned_Customers,Churn_Rate,Avg_Balance,Avg_Salary
0,High Value Customer,4351,969,22.27,"107,822.79","132,259.93"
1,Standard Value Customer,5649,1068,18.91,"52,349.43","75,312.35"


### High-Value Disengaged Customer Flag

In [20]:
# Identify high-value customers who are inactive
features_df["High_Value_Disengaged_Flag"] = np.where(
    (features_df["High_Value_Customer_Flag"] == 1) &
    (features_df["IsActiveMember"] == 0),
    1,
    0
)

features_df["High_Value_Disengaged_Status"] = features_df["High_Value_Disengaged_Flag"].map({
    0: "Not High-Value Disengaged",
    1: "High-Value Disengaged"
})

high_value_disengaged_distribution = (
    features_df["High_Value_Disengaged_Status"]
    .value_counts()
    .reset_index()
)

high_value_disengaged_distribution.columns = [
    "High-Value Disengaged Status",
    "Customer Count"
]

high_value_disengaged_distribution["Customer %"] = (
    high_value_disengaged_distribution["Customer Count"] / len(features_df) * 100
).round(2)

high_value_disengaged_distribution

,High-Value Disengaged Status,Customer Count,Customer %
0,Not High-Value Disengaged,7846,78.46
1,High-Value Disengaged,2154,21.54


### High-Value Disengaged vs Churn Validation

In [21]:
# Validate churn rate among high-value disengaged customers
high_value_disengaged_churn = (
    features_df
    .groupby("High_Value_Disengaged_Status")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Churn_Rate=("Exited", "mean"),
        Avg_Balance=("Balance", "mean"),
        Avg_Salary=("EstimatedSalary", "mean"),
        Avg_Num_Products=("NumOfProducts", "mean")
    )
    .reset_index()
)

high_value_disengaged_churn["Churn_Rate"] = (
    high_value_disengaged_churn["Churn_Rate"] * 100
).round(2)

high_value_disengaged_churn["Avg_Balance"] = high_value_disengaged_churn["Avg_Balance"].round(2)
high_value_disengaged_churn["Avg_Salary"] = high_value_disengaged_churn["Avg_Salary"].round(2)
high_value_disengaged_churn["Avg_Num_Products"] = high_value_disengaged_churn["Avg_Num_Products"].round(2)

high_value_disengaged_churn = high_value_disengaged_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

high_value_disengaged_churn

,High_Value_Disengaged_Status,Customer_Count,Churned_Customers,Churn_Rate,Avg_Balance,Avg_Salary,Avg_Num_Products
0,High-Value Disengaged,2154,626,29.06,"108,144.37","132,331.13",1.47
1,Not High-Value Disengaged,7846,1411,17.98,"67,794.53","91,238.99",1.55


### At-Risk Premium Customer Flag

In [22]:
# Define at-risk premium customers
# Logic:
# High-value customer + inactive OR single product OR already high product load
features_df["At_Risk_Premium_Flag"] = np.where(
    (features_df["High_Value_Customer_Flag"] == 1) &
    (
        (features_df["IsActiveMember"] == 0) |
        (features_df["Product_Depth_Segment"] == "Single Product") |
        (features_df["Product_Depth_Segment"] == "High Product Load")
    ),
    1,
    0
)

features_df["At_Risk_Premium_Status"] = features_df["At_Risk_Premium_Flag"].map({
    0: "Not At-Risk Premium",
    1: "At-Risk Premium Customer"
})

at_risk_premium_distribution = (
    features_df["At_Risk_Premium_Status"]
    .value_counts()
    .reset_index()
)

at_risk_premium_distribution.columns = [
    "At-Risk Premium Status",
    "Customer Count"
]

at_risk_premium_distribution["Customer %"] = (
    at_risk_premium_distribution["Customer Count"] / len(features_df) * 100
).round(2)

at_risk_premium_distribution

,At-Risk Premium Status,Customer Count,Customer %
0,Not At-Risk Premium,6548,65.48
1,At-Risk Premium Customer,3452,34.52


### At-Risk Premium vs Churn Validation

In [23]:
# Validate churn among at-risk premium customers
at_risk_premium_churn = (
    features_df
    .groupby("At_Risk_Premium_Status")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Churn_Rate=("Exited", "mean"),
        Avg_Balance=("Balance", "mean"),
        Avg_Salary=("EstimatedSalary", "mean"),
        Avg_Num_Products=("NumOfProducts", "mean")
    )
    .reset_index()
)

at_risk_premium_churn["Churn_Rate"] = (
    at_risk_premium_churn["Churn_Rate"] * 100
).round(2)

at_risk_premium_churn["Avg_Balance"] = at_risk_premium_churn["Avg_Balance"].round(2)
at_risk_premium_churn["Avg_Salary"] = at_risk_premium_churn["Avg_Salary"].round(2)
at_risk_premium_churn["Avg_Num_Products"] = at_risk_premium_churn["Avg_Num_Products"].round(2)

at_risk_premium_churn = at_risk_premium_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

at_risk_premium_churn

,At_Risk_Premium_Status,Customer_Count,Churned_Customers,Churn_Rate,Avg_Balance,Avg_Salary,Avg_Num_Products
0,At-Risk Premium Customer,3452,912,26.42,"113,780.60","129,135.43",1.33
1,Not At-Risk Premium,6548,1125,17.18,"56,824.72","84,778.09",1.63


### Salary-Balance Mismatch Flag

In [24]:
# Identify customers with high salary but low/zero banking balance
# This may indicate that the customer earns well but does not use this bank as the primary banking relationship.
features_df["Salary_Balance_Mismatch_Flag"] = np.where(
    (features_df["EstimatedSalary"] >= salary_threshold) &
    (features_df["Balance"] <= features_df["Balance"].quantile(0.25)),
    1,
    0
)

features_df["Salary_Balance_Mismatch_Status"] = features_df["Salary_Balance_Mismatch_Flag"].map({
    0: "No Salary-Balance Mismatch",
    1: "Salary-Balance Mismatch"
})

salary_balance_mismatch_distribution = (
    features_df["Salary_Balance_Mismatch_Status"]
    .value_counts()
    .reset_index()
)

salary_balance_mismatch_distribution.columns = [
    "Salary-Balance Mismatch Status",
    "Customer Count"
]

salary_balance_mismatch_distribution["Customer %"] = (
    salary_balance_mismatch_distribution["Customer Count"] / len(features_df) * 100
).round(2)

salary_balance_mismatch_distribution

,Salary-Balance Mismatch Status,Customer Count,Customer %
0,No Salary-Balance Mismatch,9109,91.09
1,Salary-Balance Mismatch,891,8.91


### Salary-Balance Mismatch vs Churn Validation

In [25]:
# Validate churn among salary-balance mismatch customers
salary_balance_mismatch_churn = (
    features_df
    .groupby("Salary_Balance_Mismatch_Status")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Churn_Rate=("Exited", "mean"),
        Avg_Balance=("Balance", "mean"),
        Avg_Salary=("EstimatedSalary", "mean")
    )
    .reset_index()
)

salary_balance_mismatch_churn["Churn_Rate"] = (
    salary_balance_mismatch_churn["Churn_Rate"] * 100
).round(2)

salary_balance_mismatch_churn["Avg_Balance"] = salary_balance_mismatch_churn["Avg_Balance"].round(2)
salary_balance_mismatch_churn["Avg_Salary"] = salary_balance_mismatch_churn["Avg_Salary"].round(2)

salary_balance_mismatch_churn = salary_balance_mismatch_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

salary_balance_mismatch_churn

,Salary_Balance_Mismatch_Status,Customer_Count,Churned_Customers,Churn_Rate,Avg_Balance,Avg_Salary
0,No Salary-Balance Mismatch,9109,1905,20.91,"83,967.38","92,806.54"
1,Salary-Balance Mismatch,891,132,14.81,0.00,"174,553.97"


### Customer Risk Segment

In [30]:
# Create customer risk segment using engagement, value, and product behavior
# Note: Exited is NOT used here because this segment should describe behavioral risk patterns.

def assign_customer_risk_segment(row):
    if row["At_Risk_Premium_Flag"] == 1:
        return "At-Risk Premium"
    elif row["High_Value_Disengaged_Flag"] == 1:
        return "High-Value Disengaged"
    elif row["IsActiveMember"] == 0 and row["NumOfProducts"] == 1:
        return "Inactive Single-Product Risk"
    elif row["Product_Depth_Segment"] == "High Product Load":
        return "High Product Load Risk"
    elif row["Product_Depth_Segment"] == "Optimal Product Depth" and row["IsActiveMember"] == 1:
        return "Sticky Active Customer"
    else:
        return "Standard Monitoring"

features_df["Customer_Risk_Segment"] = features_df.apply(assign_customer_risk_segment, axis=1)

customer_risk_distribution = (
    features_df["Customer_Risk_Segment"]
    .value_counts()
    .reset_index()
)

customer_risk_distribution.columns = ["Customer Risk Segment", "Customer Count"]

customer_risk_distribution["Customer %"] = (
    customer_risk_distribution["Customer Count"] / len(features_df) * 100
).round(2)

customer_risk_distribution

,Customer Risk Segment,Customer Count,Customer %
0,At-Risk Premium,3452,34.52
1,Standard Monitoring,2641,26.41
2,Sticky Active Customer,2446,24.46
3,Inactive Single-Product Risk,1279,12.79
4,High Product Load Risk,182,1.82


### Customer Risk Segment vs Churn Validation

In [31]:
# Validate churn rate by customer risk segment
customer_risk_churn = (
    features_df
    .groupby("Customer_Risk_Segment")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Churn_Rate=("Exited", "mean"),
        Avg_Balance=("Balance", "mean"),
        Avg_Salary=("EstimatedSalary", "mean"),
        Avg_Num_Products=("NumOfProducts", "mean")
    )
    .reset_index()
)

customer_risk_churn["Churn_Rate"] = (
    customer_risk_churn["Churn_Rate"] * 100
).round(2)

customer_risk_churn["Avg_Balance"] = customer_risk_churn["Avg_Balance"].round(2)
customer_risk_churn["Avg_Salary"] = customer_risk_churn["Avg_Salary"].round(2)
customer_risk_churn["Avg_Num_Products"] = customer_risk_churn["Avg_Num_Products"].round(2)

customer_risk_churn = customer_risk_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

customer_risk_churn

,Customer_Risk_Segment,Customer_Count,Churned_Customers,Churn_Rate,Avg_Balance,Avg_Salary,Avg_Num_Products
1,High Product Load Risk,182,145,79.67,"54,044.33","79,915.40",3.16
2,Inactive Single-Product Risk,1279,479,37.45,"72,971.53","76,797.70",1.00
0,At-Risk Premium,3452,912,26.42,"113,780.60","129,135.43",1.33
3,Standard Monitoring,2641,365,13.82,"53,801.33","74,446.45",1.50
4,Sticky Active Customer,2446,136,5.56,"51,852.94","100,468.10",2.00


### Relationship Strength Index

In [32]:
# Create Relationship Strength Index using engagement, product depth, card ownership, tenure, and balance
features_df["Relationship_Strength_Index"] = 0

# Engagement score
features_df["Relationship_Strength_Index"] += np.where(
    features_df["IsActiveMember"] == 1,
    30,
    0
)

# Product depth score
features_df["Relationship_Strength_Index"] += np.select(
    [
        features_df["NumOfProducts"] == 1,
        features_df["NumOfProducts"] == 2,
        features_df["NumOfProducts"] >= 3
    ],
    [
        10,
        30,
        5
    ],
    default=0
)

# Credit card score
features_df["Relationship_Strength_Index"] += np.where(
    features_df["HasCrCard"] == 1,
    10,
    0
)

# Tenure score
features_df["Relationship_Strength_Index"] += np.select(
    [
        features_df["Tenure"] <= 2,
        features_df["Tenure"].between(3, 5),
        features_df["Tenure"].between(6, 8),
        features_df["Tenure"] >= 9
    ],
    [
        5,
        10,
        15,
        20
    ],
    default=0
)

# Balance relationship score
features_df["Relationship_Strength_Index"] += np.select(
    [
        features_df["Balance"] == 0,
        features_df["Balance"].between(1, 100000),
        features_df["Balance"] > 100000
    ],
    [
        0,
        10,
        20
    ],
    default=0
)

# Preview RSI range
print("Minimum Relationship Strength Index:", features_df["Relationship_Strength_Index"].min())
print("Maximum Relationship Strength Index:", features_df["Relationship_Strength_Index"].max())
print("Average Relationship Strength Index:", round(features_df["Relationship_Strength_Index"].mean(), 2))

features_df[
    [
        "CustomerId",
        "Engagement_Status",
        "Product_Depth_Segment",
        "Credit_Card_Status",
        "Tenure_Segment",
        "Balance_Segment",
        "Relationship_Strength_Index"
    ]
].head()

Minimum Relationship Strength Index: 10
Maximum Relationship Strength Index: 110
Average Relationship Strength Index: 64.44


,CustomerId,Engagement_Status,Product_Depth_Segment,Credit_Card_Status,Tenure_Segment,Balance_Segment,Relationship_Strength_Index
0,15634602,Active Member,Single Product,Has Credit Card,New Customer,Zero Balance,55
1,15647311,Active Member,Single Product,No Credit Card,New Customer,Medium Balance,55
2,15619304,Inactive Member,High Product Load,Has Credit Card,Established Relationship,Premium Balance,50
3,15701354,Inactive Member,Optimal Product Depth,No Credit Card,New Customer,Zero Balance,35
4,15737888,Active Member,Single Product,Has Credit Card,New Customer,High Balance,75


### Relationship Strength Tier

In [33]:
# Create relationship strength tier
def assign_relationship_strength_tier(score):
    if score >= 80:
        return "Strong Relationship"
    elif score >= 55:
        return "Moderate Relationship"
    elif score >= 30:
        return "Weak Relationship"
    else:
        return "Very Weak Relationship"

features_df["Relationship_Strength_Tier"] = features_df["Relationship_Strength_Index"].apply(
    assign_relationship_strength_tier
)

relationship_tier_distribution = (
    features_df["Relationship_Strength_Tier"]
    .value_counts()
    .reset_index()
)

relationship_tier_distribution.columns = ["Relationship Strength Tier", "Customer Count"]

relationship_tier_distribution["Customer %"] = (
    relationship_tier_distribution["Customer Count"] / len(features_df) * 100
).round(2)

relationship_tier_distribution

,Relationship Strength Tier,Customer Count,Customer %
0,Moderate Relationship,3996,39.96
1,Strong Relationship,2873,28.73
2,Weak Relationship,2869,28.69
3,Very Weak Relationship,262,2.62


### Final Feature Dataset Column Review

In [34]:
# Review final feature-engineered dataset structure
print(f"Final Rows: {features_df.shape[0]:,}")
print(f"Final Columns: {features_df.shape[1]:,}")

features_df.columns.tolist()

Final Rows: 10,000
Final Columns: 33


['Year',
 'CustomerId',
 'Surname',
 'CreditScore',
 'Geography',
 'Gender',
 'Age',
 'Tenure',
 'Balance',
 'NumOfProducts',
 'HasCrCard',
 'IsActiveMember',
 'EstimatedSalary',
 'Exited',
 'Churn_Status',
 'Engagement_Status',
 'Credit_Card_Status',
 'Product_Depth_Segment',
 'Balance_Segment',
 'Salary_Segment',
 'Age_Segment',
 'Tenure_Segment',
 'High_Value_Customer_Flag',
 'High_Value_Customer_Status',
 'High_Value_Disengaged_Flag',
 'High_Value_Disengaged_Status',
 'At_Risk_Premium_Flag',
 'At_Risk_Premium_Status',
 'Salary_Balance_Mismatch_Flag',
 'Salary_Balance_Mismatch_Status',
 'Customer_Risk_Segment',
 'Relationship_Strength_Index',
 'Relationship_Strength_Tier']

### Engineered Features Missing Value Check

In [35]:
# Check missing values after feature engineering
feature_missing_check = features_df.isnull().sum().reset_index()
feature_missing_check.columns = ["Column", "Missing Values"]

feature_missing_check["Missing %"] = (
    feature_missing_check["Missing Values"] / len(features_df) * 100
).round(2)

feature_missing_check = feature_missing_check.sort_values(
    by="Missing Values",
    ascending=False
)

feature_missing_check

,Column,Missing Values,Missing %
0,Year,0,0.00
1,CustomerId,0,0.00
2,Surname,0,0.00
3,CreditScore,0,0.00
4,Geography,0,0.00
5,Gender,0,0.00
6,Age,0,0.00
7,Tenure,0,0.00
8,Balance,0,0.00
9,NumOfProducts,0,0.00


## Feature Engineering & Segmentation Conclusion

Feature engineering has been successfully completed for the customer engagement and retention analysis.

Key features created in this notebook include:

- Readable churn, engagement, and credit card status columns
- Product depth segmentation
- Balance, salary, age, and tenure segmentation
- High-value customer identification
- High-value disengaged customer flag
- At-risk premium customer flag
- Salary-balance mismatch flag
- Customer risk segmentation
- Relationship Strength Index
- Relationship Strength Tier

Key business observations from the engineered features:

- High-value disengaged customers show higher churn than other customers.
- At-risk premium customers have higher churn than non-risk premium customers.
- Salary-balance mismatch is useful for relationship-depth analysis, but it is not a strong churn-risk indicator in this dataset.
- High product load customers show the highest churn risk.
- Inactive single-product customers show elevated churn risk.
- Sticky active customers show the lowest churn rate.
- Relationship strength can be measured using engagement, product depth, credit card ownership, tenure, and balance behavior.

The feature-engineered dataset is now ready for deeper analytical notebooks and Streamlit dashboard development.

### Export Feature-Engineered Dataset

In [36]:
# Create processed data directory if it does not exist
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Export feature-engineered dataset
features_df.to_csv(FEATURES_DATA_PATH, index=False)

print(f"Feature-engineered dataset exported successfully to: {FEATURES_DATA_PATH}")
print(f"Exported Rows: {features_df.shape[0]:,}")
print(f"Exported Columns: {features_df.shape[1]:,}")

Feature-engineered dataset exported successfully to: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\processed\european_bank_features.csv
Exported Rows: 10,000
Exported Columns: 33
